In [0]:
# Create a text widget named "catalog" with default value "new_catalog"
dbutils.widgets.text("catalog", "new_catalog")

# Retrieve the value entered in the "catalog" widget, strip whitespace, and store in variable Catalog1
Catalog1 = dbutils.widgets.get("catalog").strip()

# Create a text widget named "schema" with default value "default_schema"
dbutils.widgets.text("schema", "default_schema")

# Retrieve the value entered in the "schema" widget, strip whitespace, and store in variable Schema1
Schema1 = dbutils.widgets.get("schema").strip()

In [0]:
import json

# Run another notebook (common_config_nb) with a 360-second timeout
# Pass parameters dynamically using the widgets values for catalog and schema
json_obj = dbutils.notebook.run(
    "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/common_config_nb",
    360,
    {"catalog_new": Catalog1, "schema_new": Schema1}
)

# The notebook returns a JSON string; parse it into a Python dictionary
config_dict = json.loads(json_obj)

# Extract specific configuration values from the dictionary
bronze_path = config_dict["bronze_path"]      # Path for bronze layer data
silver_path = config_dict["silver_path"]      # Path for silver layer data
silver_db   = config_dict["silver_db"]        # Database/schema for silver layer tables

In [0]:
%run "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/generic_functions_nb"

In [0]:
%run "/Workspace/Users/viggneshwar@gmail.com/databricks/Logistics/Project/Generic_Function/business_specific_functions"

In [0]:
# Read raw staff data file 1 from the bronze layer (Delta format)
staff_df1_raw = read_file(file_type='delta', path=f'{bronze_path}/staff_file1_raw')

# Read raw staff data file 2 from the bronze layer (Delta format)
staff_df2_raw = read_file(file_type='delta', path=f'{bronze_path}/staff_file2_raw')

# Read raw logistics shipment data from the bronze layer (Delta format)
logistics_shipment_df_raw = read_file(file_type='delta', path=f'{bronze_path}/logistics_shipment_raw')

# Read raw master city reference data from the bronze layer (Delta format)
master_city_df_raw = read_file(file_type='delta', path=f'{bronze_path}/master_city_raw')

In [0]:
staff_df1_raw.show(5)

In [0]:
# Add a new column "data_source" to staff_df1_raw with default value "source1"
staff_df1_raw = add_column_with_default(
    df=staff_df1_raw,
    column_name='data_source',
    default_value='source1'
)

# Add a new column "data_source" to staff_df2_raw with default value "source2"
staff_df2_raw = add_column_with_default(
    df=staff_df2_raw,
    column_name='data_source',
    default_value='source2'
)

# Merge the two staff DataFrames into one consolidated DataFrame
# The third parameter (True) indicates that duplicates should be handled (depending on your merge_df implementation)
staff_df_raw = merge_df(staff_df1_raw, staff_df2_raw, True)

In [0]:
staff_df_raw.show(5)

In [0]:
# First cleansing step:
# - Remove duplicate rows based on 'shipment_id'
# - Drop rows with nulls in 'shipment_id' or 'role' columns
# - Use 'any' strategy: if any of the specified columns are null, drop the row
staff_df_cleansed = cleansing_func(
    staff_df_raw,
    duplicatedatacolumns=['shipment_id'],
    nulldropcolumns=['shipment_id', 'role'],
    nullstrategy='any'
)

# Second cleansing step:
# - Drop rows where both 'first_name' and 'last_name' are null
# - Use 'all' strategy: drop only if all specified columns are null
staff_df_cleansed1 = cleansing_func(
    staff_df_cleansed,
    nulldropcolumns=['first_name', 'last_name'],
    nullstrategy='all'
)

In [0]:
# Apply staff standardisation rules:
# - Lowercase role
# - Initcap hub_location
# - Convert age/shipment_id to integers
# - Rename columns for clarity
staff_standard_df = staff_data_standardisation_func(staff_df_cleansed1)

# Apply logistics shipment standardisation rules:
# - Add domain, ingestion timestamp
# - Normalize vehicle type casing
# - Convert shipment_date to proper date format
# - Round shipment_cost, cast weight
# - Ensure expedited flag is boolean
# Apply the standardisation function to your raw DataFrame
logistics_shipment_std_df = logistics_shipment_data_standardisation_func(logistics_shipment_df_raw)

# Inspect the schema of the standardized DataFrame
logistics_shipment_std_df.printSchema()

In [0]:
logistics_shipment_std_df.show(5)

In [0]:
# Enrich staff data:
# - Adds load timestamp
# - Creates full_name from first and last names
# - Selects relevant business columns
staff_enriched_df = staff_data_enrichedment_func(staff_standard_df)

# Enrich logistics shipment data:
# - Adds route segment and lane
# - Creates vehicle identifier
# - Derives shipment year/month/day
# - Flags weekend shipments
# - Flags expedited shipments based on status
# - Calculates cost per kg safely
# - Adds days since shipment, tax amount
# - Splits order_id into prefix and sequence
logistics_shipment_enriched_df = logistics_shipment_data_enrichment_func(logistics_shipment_std_df)
logistics_shipment_enriched_df.display()

In [0]:
# Apply staff customisation rules:
# - Adds projected_bonus using bonus_udf (based on role and age)
# - Masks full_name for privacy using mask_string_udf
staff_customized_df = staff_data_customize_func(staff_enriched_df)

In [0]:
# Write the customized staff DataFrame into the silver layer table
write_file(
    staff_customized_df,
    file_type='table',
    table=f'{silver_db}.staff_silver_tbl'
)

# Write the enriched logistics shipment DataFrame into the silver layer table
write_file(
    logistics_shipment_enriched_df,
    file_type='table',
    table=f'{silver_db}.logistics_shipment_silver_tbl'
)

In [0]:
# Write the raw master city DataFrame into the silver layer table
write_file(
    master_city_df_raw,
    file_type='table',
    table=f'{silver_db}.master_city_tbl'
)